Goal: export gold graphs for CHIA struct_eval_200

Inputs: data/processed/chia_struct_eval_200.jsonl

Output: data/processed/chia_struct_eval_200_gold_graph.jsonl

In [3]:
from datasets import load_dataset
import json
from pathlib import Path
import json

# Find project root
current = Path.cwd().resolve()
ROOT = current.parent if current.name == "notebooks" else current

processed_path = ROOT / "data" / "processed" / "chia_struct_eval_200.jsonl"
out_path = ROOT / "data" / "processed" / "chia_struct_eval_200_gold_graph.jsonl"

print("ROOT:", ROOT)
print("Input:", processed_path)
print("Output:", out_path)

ROOT: C:\Users\JUlloa\Documents\MSc Statistics and Data Science\Master Thesis\clinical-trial-eligibility-rule-verification
Input: C:\Users\JUlloa\Documents\MSc Statistics and Data Science\Master Thesis\clinical-trial-eligibility-rule-verification\data\processed\chia_struct_eval_200.jsonl
Output: C:\Users\JUlloa\Documents\MSc Statistics and Data Science\Master Thesis\clinical-trial-eligibility-rule-verification\data\processed\chia_struct_eval_200_gold_graph.jsonl


In [4]:
if not processed_path.exists():
    raise FileNotFoundError(processed_path)

# Load selected IDs
ids = []

with open(processed_path, "r", encoding="utf-8") as f:
    for line in f:
        ids.append(json.loads(line)["chia_id"])

print("Selected rows:", len(ids))
print("First IDs:", ids[:10])

Selected rows: 200
First IDs: ['NCT03416413_exc', 'NCT03416413_inc', 'NCT02121145_exc', 'NCT02121145_inc', 'NCT02243553_exc', 'NCT02243553_inc', 'NCT03113253_exc', 'NCT03113253_inc', 'NCT00917891_exc', 'NCT00917891_inc']


In [5]:
# Load CHIA
ds = load_dataset("bigbio/chia", "chia_bigbio_kb", split="train")
by_id = {ex["id"]: ex for ex in ds}

missing = [cid for cid in ids if cid not in by_id]
print("Missing IDs:", len(missing))

if missing:
    raise RuntimeError(f"Missing IDs: {missing[:5]}")

Using the latest cached version of the dataset since bigbio/chia couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'chia_bigbio_kb' at C:\Users\JUlloa\.cache\huggingface\datasets\bigbio___chia\chia_bigbio_kb\1.0.0\36e5df0d60dfc5152cd22a807ade73f135105008 (last modified on Mon Feb 23 14:00:31 2026).


Missing IDs: 0


convert one CHIA example to a “gold graph”

In [6]:
# Convert to gold graph
def to_gold_graph(ex):
    nodes = []

    for e in ex["entities"]:
        nodes.append({
            "id": e["id"],
            "type": e["type"],
            "text": e.get("text", []),
            "offsets": e.get("offsets", []),
            "normalized": e.get("normalized", []),
        })

    edges = []

    for r in ex["relations"]:
        edges.append({
            "id": r.get("id", None),
            "type": r["type"],
            "arg1_id": r["arg1_id"],
            "arg2_id": r["arg2_id"],
        })

    return {
        "chia_id": ex["id"],
        "document_id": ex["document_id"],
        "nodes": nodes,
        "edges": edges,
    }
# Export
with open(out_path, "w", encoding="utf-8") as f:
    for cid in ids:
        g = to_gold_graph(by_id[cid])
        f.write(json.dumps(g) + "\n")

print("Saved:", out_path)

Saved: C:\Users\JUlloa\Documents\MSc Statistics and Data Science\Master Thesis\clinical-trial-eligibility-rule-verification\data\processed\chia_struct_eval_200_gold_graph.jsonl


In [7]:
# Final check
with open(out_path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

print("Gold graph rows:", len(rows))
print("First row:", rows[0]["chia_id"], "nodes:", len(rows[0]["nodes"]), "edges:", len(rows[0]["edges"]))

Gold graph rows: 200
First row: NCT03416413_exc nodes: 14 edges: 5
